# VoiceIQ — Data Cleaning & Entity Reconciliation

Blends three independent public datasets into the four-table schema (`sql/01_schema.sql`). Since the sources share no key, this notebook also does identity reconciliation and writes warehouse-ready CSVs to `data/processed/`. Every derived/modeled field is logged in `docs/architecture.md`.

In [1]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)
REFERENCE_DATE = pd.Timestamp("2025-06-30")
WINDOW_START = REFERENCE_DATE - pd.DateOffset(months=24)

RAW = "../data/raw"
OUT = "../data/processed"

## 1. Customers — Telco Customer Churn

Real: `tenure`, `MonthlyCharges`, `Churn`, `Contract`. Everything else is a documented derivation (see `docs/architecture.md`).

In [2]:
telco = pd.read_csv(f"{RAW}/telco_customer_churn.csv")

telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0.0)

n = len(telco)

customers = pd.DataFrame({
    "customer_id": telco["customerID"],
    "tenure_months": telco["tenure"].astype(int),
    "monthly_revenue": telco["MonthlyCharges"].round(2),
    "is_churned": telco["Churn"].eq("Yes"),
})

customers["signup_date"] = customers["tenure_months"].apply(
    lambda m: (REFERENCE_DATE - pd.DateOffset(months=int(m))).date()
)

def make_churn_date(signup_date, is_churned):
    if not is_churned:
        return pd.NaT
    days_back = int(RNG.integers(1, 181))
    cd = REFERENCE_DATE - pd.Timedelta(days=days_back)
    return max(cd, pd.Timestamp(signup_date)).date()

customers["churn_date"] = [
    make_churn_date(sd, ch) for sd, ch in zip(customers["signup_date"], customers["is_churned"])
]

COUNTRIES = ["United States", "India", "United Kingdom", "Canada", "Australia", "Germany"]
COUNTRY_W = [0.42, 0.16, 0.14, 0.10, 0.10, 0.08]
customers["country"] = RNG.choice(COUNTRIES, size=n, p=COUNTRY_W)

q1, q2 = customers["monthly_revenue"].quantile([1/3, 2/3])
def segment(rev):
    if rev <= q1:
        return "SMB"
    if rev <= q2:
        return "Mid-Market"
    return "Enterprise"
customers["customer_segment"] = customers["monthly_revenue"].apply(segment)

PLAN_MAP = {"Month-to-month": "Starter", "One year": "Growth", "Two year": "Enterprise"}
customers["subscription_plan"] = telco["Contract"].map(PLAN_MAP)

customers = customers[[
    "customer_id", "signup_date", "country", "customer_segment",
    "subscription_plan", "monthly_revenue", "tenure_months", "is_churned", "churn_date",
]]

print("customers:", customers.shape)
customers.head(3)

customers: (7043, 9)


,customer_id,signup_date,country,customer_segment,subscription_plan,monthly_revenue,tenure_months,is_churned,churn_date
0,7590-VHVEG,2025-05-30,United States,SMB,Starter,29.85,1,False,NaT
1,5575-GNVDE,2022-08-30,Germany,Mid-Market,Growth,56.95,34,False,NaT
2,3668-QPYBK,2025-04-30,United States,Mid-Market,Starter,53.85,2,True,2025-06-13


In [3]:
print(customers["customer_segment"].value_counts())
print()
print(customers["is_churned"].value_counts())

customer_segment
SMB           2351
Enterprise    2347
Mid-Market    2345
Name: count, dtype: int64

is_churned
False    5174
True     1869
Name: count, dtype: int64


## 2. Support Tickets

`First Response Time`/`Time to Resolution` cluster inside a single 48-hour window across all 8,469 rows — not usable timestamps, so they're dropped. `Ticket Priority`, `Ticket Type`, `Ticket Status`, and `Customer Satisfaction Rating` are kept as-is.

In [4]:
support = pd.read_csv(f"{RAW}/support_tickets_raw.csv")

STATUS_MAP = {
    "Closed": "Closed",
    "Open": "Open",
    "Pending Customer Response": "In Progress",
}
support["status"] = support["Ticket Status"].map(STATUS_MAP)
support["status"].value_counts()

status
In Progress    2881
Open           2819
Closed         2769
Name: count, dtype: int64

### Reconciling `customer_id` + event dates

Tickets share no key with customers, so `customer_id` is a weighted random draw (favoring churned/short-tenure customers). Sampling dates uniformly per customer skews volume toward the window's end for short-tenure customers, so the sampler below fixes target row counts per calendar month first, then samples eligible customers for that month.

In [5]:
signup_lookup = customers.set_index("customer_id")["signup_date"]

MONTHS = pd.period_range(WINDOW_START, REFERENCE_DATE, freq="M")
signup_dt = pd.to_datetime(customers["signup_date"])
churn_dt = pd.to_datetime(customers["churn_date"])
cust_ids_arr = customers["customer_id"].to_numpy()
signup_by_id = pd.to_datetime(signup_lookup)

def month_eligible_customer_sampler(total_count, growth_low=0.75, growth_high=1.25, weights=None):
    """Distribute `total_count` events across MONTHS on a gentle linear growth
    curve, sampling only customers eligible (active) in each month."""
    n_months = len(MONTHS)
    growth = np.linspace(growth_low, growth_high, n_months)
    growth = growth / growth.sum()
    target_counts = np.round(growth * total_count).astype(int)
    target_counts[-1] += total_count - target_counts.sum()

    out_customers, out_dates = [], []
    for month, cnt in zip(MONTHS, target_counts):
        if cnt <= 0:
            continue
        month_start = month.start_time
        upper_bound = min(month.end_time, REFERENCE_DATE)
        eligible = (signup_dt <= month.end_time) & (churn_dt.isna() | (churn_dt >= month_start))
        elig_ids = cust_ids_arr[eligible.to_numpy()]
        if len(elig_ids) == 0:
            continue
        p = None
        if weights is not None:
            w = weights[eligible.to_numpy()]
            p = w / w.sum()
        picked = RNG.choice(elig_ids, size=cnt, p=p)

        lower = np.maximum(
            np.datetime64(month_start),
            signup_by_id.reindex(picked).to_numpy().astype("datetime64[ns]"),
        )
        span_s = np.maximum((np.datetime64(upper_bound) - lower) / np.timedelta64(1, "s"), 0)
        offsets = (RNG.random(cnt) * span_s).astype("int64")
        dates = lower + offsets.astype("timedelta64[s]")

        out_customers.append(picked)
        out_dates.append(dates)

    customer_ids = np.concatenate(out_customers)
    dates = np.concatenate(out_dates)
    order = RNG.permutation(len(customer_ids))
    return customer_ids[order], dates[order]

ticket_weights = (1.0 + customers["is_churned"].astype(float) * 2.0 +
                  (24 - customers["tenure_months"]).clip(lower=0) / 24.0).to_numpy()
picked_customers, picked_dates = month_eligible_customer_sampler(len(support), weights=ticket_weights)
support["customer_id"] = picked_customers
support["ticket_created_at"] = pd.to_datetime(picked_dates)

support["ticket_created_at"].dt.to_period("M").value_counts().sort_index()

ticket_created_at
2023-06    254
2023-07    261
2023-08    268
2023-09    275
2023-10    282
2023-11    289
2023-12    296
2024-01    303
2024-02    311
2024-03    318
2024-04    325
2024-05    332
2024-06    339
2024-07    346
2024-08    353
2024-09    360
2024-10    367
2024-11    374
2024-12    381
2025-01    388
2025-02    395
2025-03    402
2025-04    409
2025-05    416
2025-06    425
Freq: M, Name: count, dtype: int64

### Resolution time + CSAT

`resolution_time` is modeled from `Ticket Priority` via SLA-calibrated lognormal bands. The real CSAT rating is then nudged by how far that modeled duration deviated from its SLA — otherwise the two would show no correlation at all.

In [6]:
SLA_HOURS = {"Critical": 4, "High": 12, "Medium": 30, "Low": 60}

def sample_resolution_hours(priority):
    mu_hours = SLA_HOURS[priority]
    sigma = 0.6
    return float(RNG.lognormal(mean=np.log(mu_hours), sigma=sigma))

support["_resolution_hours"] = support["Ticket Priority"].map(sample_resolution_hours)
support["ticket_resolved_at"] = np.where(
    support["status"] == "Closed",
    support["ticket_created_at"] + pd.to_timedelta(support["_resolution_hours"], unit="h"),
    pd.NaT,
)
support["ticket_resolved_at"] = pd.to_datetime(support["ticket_resolved_at"]).clip(upper=REFERENCE_DATE)

sla_expected = support["Ticket Priority"].map(SLA_HOURS)
sla_deviation = ((support["_resolution_hours"] - sla_expected) / sla_expected).clip(-2, 2)
adjusted_csat = support["Customer Satisfaction Rating"] - 2.2 * sla_deviation
support["_csat_rating"] = adjusted_csat.round().clip(1, 5).astype("Int64")
support.loc[support["status"] != "Closed", "_csat_rating"] = np.nan

tickets = pd.DataFrame({
    "customer_id": support["customer_id"],
    "category": support["Ticket Type"],
    "priority": support["Ticket Priority"],
    "status": support["status"],
    "ticket_created_at": support["ticket_created_at"],
    "ticket_resolved_at": support["ticket_resolved_at"],
    "csat_rating": support["_csat_rating"],
})

resolved_hours = (tickets["ticket_resolved_at"] - tickets["ticket_created_at"]).dt.total_seconds() / 3600
print("support_tickets:", tickets.shape)
print(tickets["status"].value_counts())
print("\nresolution hours, closed tickets:\n", resolved_hours.describe())
print("\ncorr(csat_rating, resolution_hours):",
      pd.concat([tickets["csat_rating"], resolved_hours.rename("res_hrs")], axis=1).dropna().corr().iloc[0, 1])
assert not (resolved_hours < 0).any(), "resolved before created — integrity violation"
assert ((tickets["status"] == "Closed") & tickets["ticket_resolved_at"].isna()).sum() == 0
assert ((tickets["status"] != "Closed") & tickets["ticket_resolved_at"].notna()).sum() == 0

support_tickets: (8469, 7)
status
In Progress    2881
Open           2819
Closed         2769
Name: count, dtype: int64

resolution hours, closed tickets:
 count    2769.000000
mean       30.772471
std        38.127692
min         0.603618
25%         6.629510
50%        17.480083
75%        40.230354
max       558.824489
dtype: float64

corr(csat_rating, resolution_hours): -0.29280050010667846


## 3. Feedback — Google Play Store reviews

Real review text (dropping rows with none). `customer_id`/`created_at`/`channel` are reconciled the same way as tickets. No star rating in this source, so `source_rating` stays null.

In [7]:
gplay = pd.read_csv(f"{RAW}/playstore_user_reviews.csv")
gplay = gplay.dropna(subset=["Translated_Review"]).copy()
gplay = gplay[gplay["Translated_Review"].str.strip() != ""]

m = len(gplay)
fb_customers, fb_dates = month_eligible_customer_sampler(m, weights=None)
gplay["customer_id"] = fb_customers
gplay["created_at"] = pd.to_datetime(fb_dates)

CHANNELS = ["App Store", "In-App Survey", "Support Chat", "Email"]
CHANNEL_W = [0.5, 0.2, 0.2, 0.1]
gplay["feedback_channel"] = RNG.choice(CHANNELS, size=m, p=CHANNEL_W)

feedback = pd.DataFrame({
    "customer_id": gplay["customer_id"],
    "feedback_text": gplay["Translated_Review"].str.strip(),
    "feedback_channel": gplay["feedback_channel"],
    "source_rating": np.nan,
    "created_at": gplay["created_at"],
})
print("feedback:", feedback.shape)
feedback["feedback_channel"].value_counts()

feedback: (37427, 5)


feedback_channel
App Store        18745
Support Chat      7521
In-App Survey     7387
Email             3774
Name: count, dtype: int64

## 4. Surveys — modeled NPS + relationship CSAT

No public dataset exposes customer-linked NPS at this scale, so it's generated: one quarterly survey wave per active customer, calibrated to SaaS NPS benchmarks (~30-40), then pulled down near churn and after high-severity/slow tickets — so later at-risk analysis has a real signal to find.

In [8]:
survey_rows = []
tickets_by_cust = tickets.groupby("customer_id")
QUARTERS = pd.period_range(WINDOW_START, REFERENCE_DATE, freq="Q")

for q in QUARTERS:
    q_start = q.start_time
    q_end = min(q.end_time, REFERENCE_DATE)
    eligible = customers[(signup_dt <= q_end) & (churn_dt.isna() | (churn_dt >= q_start))]

    for _, cust in eligible.iterrows():
        cid = cust["customer_id"]
        lo = max(q_start, pd.Timestamp(cust["signup_date"]))
        hi = q_end
        d = lo if lo >= hi else lo + pd.Timedelta(seconds=int(RNG.integers(0, int((hi - lo).total_seconds()))))
        end = REFERENCE_DATE if not cust["is_churned"] else pd.Timestamp(cust["churn_date"])

        base = RNG.normal(loc=8.45, scale=1.5)

        if cust["is_churned"] and (end - d).days < 120:
            base -= 1.6

        if cid in tickets_by_cust.groups:
            recent = tickets_by_cust.get_group(cid)
            recent = recent[(recent["ticket_created_at"] <= d) &
                             (recent["ticket_created_at"] > d - pd.Timedelta(days=90))]
            if len(recent):
                crit_share = (recent["priority"].isin(["Critical", "High"])).mean()
                avg_res = recent["ticket_resolved_at"].sub(recent["ticket_created_at"]).dt.total_seconds().div(3600).mean()
                base -= crit_share * 1.2
                if pd.notna(avg_res):
                    base -= min(avg_res / 72.0, 1.2)

        nps = int(np.clip(round(base + RNG.normal(0, 0.6)), 0, 10))
        csat = int(np.clip(round(1 + (nps / 10) * 4 + RNG.normal(0, 0.6)), 1, 5))
        survey_rows.append((cid, d.date(), nps, csat))

surveys = pd.DataFrame(survey_rows, columns=["customer_id", "survey_date", "nps_score", "csat_score"])

promoters = (surveys["nps_score"] >= 9).mean()
detractors = (surveys["nps_score"] <= 6).mean()
print("surveys:", surveys.shape)
print(f"Promoter%={promoters:.1%}  Detractor%={detractors:.1%}  NPS={100*(promoters - detractors):.1f}")

s2 = surveys.merge(customers[["customer_id", "is_churned"]], on="customer_id")
print("\nAvg NPS, churned vs. retained customers (sanity check on the designed effect):")
print(s2.groupby("is_churned")["nps_score"].mean())

surveys: (45650, 4)
Promoter%=44.4%  Detractor%=15.2%  NPS=29.1

Avg NPS, churned vs. retained customers (sanity check on the designed effect):
is_churned
False    8.237239
True     7.561212
Name: nps_score, dtype: float64


## 5. Referential & temporal integrity checks

In [9]:
valid_ids = set(customers["customer_id"])
assert (~surveys["customer_id"].isin(valid_ids)).sum() == 0, "orphan survey rows"
assert (~feedback["customer_id"].isin(valid_ids)).sum() == 0, "orphan feedback rows"
assert (~tickets["customer_id"].isin(valid_ids)).sum() == 0, "orphan ticket rows"

sl = pd.to_datetime(customers.set_index("customer_id")["signup_date"])
for name, df, col in [("tickets", tickets, "ticket_created_at"), ("feedback", feedback, "created_at")]:
    merged = df.merge(sl.rename("signup_date"), left_on="customer_id", right_index=True)
    bad = (merged[col] < merged["signup_date"]).sum()
    assert bad == 0, f"{name}: {bad} rows precede customer signup_date"

print("All referential and temporal integrity checks passed.")

All referential and temporal integrity checks passed.


## 6. Write processed outputs

`nps_category`/`resolution_time` aren't written — Postgres computes them as generated columns.

In [10]:
import os
os.makedirs(OUT, exist_ok=True)
customers.to_csv(f"{OUT}/customers.csv", index=False)
surveys.to_csv(f"{OUT}/surveys.csv", index=False)
feedback.to_csv(f"{OUT}/feedback.csv", index=False)
tickets.to_csv(f"{OUT}/support_tickets.csv", index=False)

for name, df in [("customers", customers), ("surveys", surveys), ("feedback", feedback), ("support_tickets", tickets)]:
    print(f"{name:16s} -> {len(df):>6,} rows  ->  data/processed/{name}.csv")

customers        ->  7,043 rows  ->  data/processed/customers.csv
surveys          -> 45,650 rows  ->  data/processed/surveys.csv
feedback         -> 37,427 rows  ->  data/processed/feedback.csv
support_tickets  ->  8,469 rows  ->  data/processed/support_tickets.csv
